# ScribeAI Evaluation Notebook

This notebook provides an interactive environment for:
1. Loading and exploring SOAP generation results
2. Visualizing ROUGE/BLEU score distributions
3. Comparing generation modes (OpenAI vs Anthropic vs Demo)
4. Analyzing validation warning patterns
5. Identifying the best and worst generated notes

**Prerequisites:**
```bash
pip install matplotlib pandas seaborn jupyter
```

**Generate evaluation data first:**
```bash
python evaluation/scripts/batch_evaluate.py \
    --dataset data/synthetic/synthetic_soap_examples.json \
    --output evaluation/reports/results.json
```

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configure plots
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('Imports ready!')

In [ ]:
# Load evaluation results
RESULTS_PATH = '../../evaluation/reports/results.json'

with open(RESULTS_PATH) as f:
    report = json.load(f)

print(f"Report generated: {report['generated_at']}")
print(f"Mode: {report['configuration']['mode']}")
print(f"Total examples: {report['configuration']['total_examples']}")

stats = report['aggregate_statistics']
print('\n=== Aggregate Statistics ===')
for k, v in stats.items():
    print(f'  {k}: {v}')

In [ ]:
# Build a DataFrame from results
rows = []
for r in report['results']:
    if r.get('error'):
        continue
    
    gen = r.get('generation', {})
    meta = gen.get('metadata', {})
    eval_data = r.get('evaluation', {})
    scores = eval_data.get('scores', {})
    section_scores = scores.get('section_scores') or {}
    
    rows.append({
        'id': r['id'],
        'chief_complaint': r.get('chief_complaint', 'Unknown'),
        'mode': meta.get('mode', 'unknown'),
        'model': meta.get('model', 'unknown'),
        'processing_time_ms': meta.get('processing_time_ms'),
        'sections_populated': meta.get('sections_populated'),
        'transcript_word_count': meta.get('transcript_word_count'),
        'note_word_count': meta.get('note_word_count'),
        'n_warnings': len(gen.get('warnings', [])),
        'has_reference': r.get('has_reference', False),
        'rouge_1': scores.get('rouge_1'),
        'rouge_2': scores.get('rouge_2'),
        'rouge_l': scores.get('rouge_l'),
        'bleu': scores.get('bleu'),
        'heuristic': section_scores.get('heuristic_completeness'),
    })

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} results into DataFrame')
df.head()

In [ ]:
# Sections populated distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Processing time
axes[0].hist(df['processing_time_ms'].dropna(), bins=20, color='#0ea5e9', edgecolor='white')
axes[0].set_xlabel('Processing Time (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Generation Time Distribution')

# Sections populated
section_counts = df['sections_populated'].value_counts().sort_index()
axes[1].bar(section_counts.index, section_counts.values, color='#10b981', edgecolor='white')
axes[1].set_xlabel('Sections Populated')
axes[1].set_ylabel('Count')
axes[1].set_title('SOAP Sections Populated')
axes[1].set_xticks([1, 2, 3, 4])

# Warnings
warn_counts = df['n_warnings'].value_counts().sort_index()
axes[2].bar(warn_counts.index, warn_counts.values, color='#f59e0b', edgecolor='white')
axes[2].set_xlabel('Number of Warnings')
axes[2].set_ylabel('Count')
axes[2].set_title('Validation Warnings per Note')

plt.tight_layout()
plt.savefig('../../evaluation/reports/generation_stats.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROUGE scores (for examples with reference notes)
ref_df = df[df['has_reference'] & df['rouge_l'].notna()]

if len(ref_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ROUGE score bars
    metrics = ['rouge_1', 'rouge_2', 'rouge_l']
    colors = ['#3b82f6', '#8b5cf6', '#06b6d4']
    means = [ref_df[m].mean() for m in metrics]
    stds = [ref_df[m].std() for m in metrics]
    
    bars = axes[0].bar(['ROUGE-1', 'ROUGE-2', 'ROUGE-L'], means, color=colors, 
                        yerr=stds, capsize=5, edgecolor='white')
    axes[0].set_ylabel('Score')
    axes[0].set_title('Average ROUGE Scores')
    axes[0].set_ylim(0, 1)
    for bar, mean in zip(bars, means):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f'{mean:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Per-example ROUGE-L
    axes[1].scatter(range(len(ref_df)), ref_df['rouge_l'].values, 
                   color='#0ea5e9', s=60, zorder=3)
    axes[1].axhline(y=ref_df['rouge_l'].mean(), color='#ef4444', 
                   linestyle='--', label=f'Mean: {ref_df["rouge_l"].mean():.3f}')
    axes[1].set_xlabel('Example Index')
    axes[1].set_ylabel('ROUGE-L Score')
    axes[1].set_title('ROUGE-L per Example')
    axes[1].legend()
    axes[1].set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig('../../evaluation/reports/rouge_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'Examples with reference notes: {len(ref_df)}')
    print(f'Average ROUGE-L: {ref_df["rouge_l"].mean():.4f}')
    print(f'Average ROUGE-1: {ref_df["rouge_1"].mean():.4f}')
else:
    print('No examples with reference notes found. Run evaluation with reference notes to see ROUGE scores.')

In [ ]:
# Show best and worst performing notes by ROUGE-L
if len(ref_df) > 0:
    print('=== TOP 3 HIGHEST ROUGE-L NOTES ===')
    for _, row in ref_df.nlargest(3, 'rouge_l').iterrows():
        print(f'  {row["id"]} | {row["chief_complaint"]} | ROUGE-L: {row["rouge_l"]:.4f}')

    print('\n=== BOTTOM 3 LOWEST ROUGE-L NOTES ===')
    for _, row in ref_df.nsmallest(3, 'rouge_l').iterrows():
        print(f'  {row["id"]} | {row["chief_complaint"]} | ROUGE-L: {row["rouge_l"]:.4f}')

In [ ]:
# Summary statistics table
print('=== SUMMARY ===')
summary = df[['processing_time_ms', 'sections_populated', 'n_warnings', 
              'transcript_word_count', 'note_word_count']].describe().round(2)
print(summary.to_string())